In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


In [ ]:
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.cluster import KMeans
from sklearn.ensemble import RandomForestClassifier


In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    roc_curve,
    auc
)


In [ ]:
df_original = pd.read_excel("Telco_customer_churn.xlsx")
df = df_original.copy()
df.head()


In [ ]:
df.shape


In [ ]:
df.info()


In [ ]:
df.isnull().sum()


In [ ]:
df['Total Charges'] = pd.to_numeric(df['Total Charges'], errors='coerce')
df['Total Charges'].fillna(df['Total Charges'].median(), inplace=True)


In [ ]:
if 'Churn Reason' in df.columns:
    df['Churn Reason'] = df['Churn Reason'].fillna('No Churn')


In [ ]:
sns.countplot(x='Churn Label', data=df_original)
plt.show()


In [ ]:
sns.countplot(x='Gender', hue='Churn Label', data=df_original)
plt.show()


In [ ]:
sns.countplot(x='Contract', hue='Churn Label', data=df_original)
plt.xticks(rotation=30)
plt.show()


In [ ]:
sns.histplot(data=df_original, x='Monthly Charges', hue='Churn Label', kde=True)
plt.show()


In [ ]:
sns.boxplot(x='Churn Label', y='Tenure Months', data=df_original)
plt.show()


In [ ]:
drop_columns = [
'CustomerID','Count','Country','State','Zip Code',
'Lat Long','Latitude','Longitude',
'Churn Score','Churn Reason','CLTV'
]

existing = [c for c in drop_columns if c in df.columns]
df = df.drop(columns=existing)


In [ ]:
le = LabelEncoder()

for col in df.columns:
    if df[col].dtype == 'object':
        df[col] = le.fit_transform(df[col].astype(str))

df.head()


In [ ]:
plt.figure(figsize=(15,10))
sns.heatmap(df.corr(), cmap='coolwarm')
plt.show()


In [ ]:
seg_features = ['Tenure Months','Monthly Charges','Total Charges']
seg_data = df[seg_features]


In [ ]:
scaler = StandardScaler()
seg_scaled = scaler.fit_transform(seg_data)


In [ ]:
wcss = []

for i in range(1,11):
    km = KMeans(n_clusters=i, random_state=42, n_init=10)
    km.fit(seg_scaled)
    wcss.append(km.inertia_)

plt.plot(range(1,11), wcss)
plt.show()


In [ ]:
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
df['Customer Segment'] = kmeans.fit_predict(seg_scaled)


In [ ]:
df.groupby('Customer Segment')[seg_features].mean()


In [ ]:
sns.scatterplot(
    x='Monthly Charges',
    y='Total Charges',
    hue='Customer Segment',
    data=df
)
plt.show()


In [ ]:
X = df.drop(['Churn Value','Churn Label'], axis=1, errors='ignore')
y = df['Churn Value']


In [ ]:
X_train,X_test,y_train,y_test = train_test_split(
    X,y,test_size=0.2,random_state=42,stratify=y
)


In [ ]:
param_grid = {
'n_estimators':[100,200,300],
'max_depth':[5,10,15],
'min_samples_split':[2,5,10],
'min_samples_leaf':[1,2,4]
}


In [ ]:
rf = RandomForestClassifier(
    random_state=42,
    class_weight='balanced'
)

grid = GridSearchCV(
    rf,
    param_grid,
    cv=5,
    scoring='f1',
    n_jobs=-1
)


In [ ]:
grid.fit(X_train,y_train)


In [ ]:
grid.best_params_


In [ ]:
rf_tuned = grid.best_estimator_


In [ ]:
y_pred = rf_tuned.predict(X_test)


In [ ]:
accuracy_score(y_test,y_pred)


In [ ]:
precision_score(y_test,y_pred)


In [ ]:
recall_score(y_test,y_pred)


In [ ]:
f1_score(y_test,y_pred)


In [ ]:
print(classification_report(y_test,y_pred))


In [ ]:
cm = confusion_matrix(y_test,y_pred)

sns.heatmap(cm,annot=True,fmt='d')
plt.show()


In [ ]:
y_prob_test = rf_tuned.predict_proba(X_test)[:,1]

fpr,tpr,thresholds = roc_curve(y_test,y_prob_test)
roc_auc = auc(fpr,tpr)

plt.plot(fpr,tpr,label=f'AUC={roc_auc:.3f}')
plt.legend()
plt.show()


In [ ]:
importance = pd.DataFrame({
'Feature':X.columns,
'Importance':rf_tuned.feature_importances_
})

importance.sort_values(
by='Importance',
ascending=False
).head(20)


In [ ]:
df['Churn Probability'] = rf_tuned.predict_proba(X)[:,1]


In [ ]:
conditions = [
df['Churn Probability'] >= 0.70,
(df['Churn Probability'] >= 0.40) & (df['Churn Probability'] < 0.70),
df['Churn Probability'] < 0.40
]

choices = ['High Risk','Medium Risk','Low Risk']

df['Risk Category'] = np.select(
conditions,
choices,
default='Low Risk'
)


In [ ]:
df[['Churn Probability','Risk Category']].head()


In [ ]:
high_risk = df[df['Risk Category']=='High Risk']
high_risk.head(20)


In [ ]:
df.to_csv('customer_churn_results.csv',index=False)
